In [ ]:
%%writefile submission.py
# ！！！PLEASE DO NOT EDIT IT ！！！
# This command will save the code you wrote as a submission.py file in the same directory, for the test script to read.

# packages
import os
import sys
import time
import json
from typing import Dict, List, Tuple, Optional, Iterable, Iterator
from collections import defaultdict, Counter
import heapq
from dataclasses import dataclass
import tqdm
import psutil
import tracemalloc

# my answer functions

def find_data_file():
    paths = [os.path.expanduser("~/tinystories_bpe/data/TinyStoriesV2-GPT4-valid.txt"),  ]
    
    found_files = []
    
    if found_files:
        found_files.sort(key=lambda x: x[1], reverse=True)
        return found_files[0][0]

def BPETrainer__init__(self, special_tokens: List[str] = None):
        self.special_tokens = special_tokens or []
        self.vocab = {} 
        self.merges = [] 
        self.reverse_vocab = {}

def train(self, input_path: str, vocab_size: int) -> Tuple[Dict[int, bytes], List[Tuple[bytes, bytes]]]:
        
        self._init_base_vocab()
        with open(input_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        words = text.split()
        word_freqs = Counter(words)

        byte_word_freqs = {}
        for word, freq in word_freqs.items():
            try:
                byte_word = word.encode('utf-8')
                if all(0 <= b <= 255 for b in byte_word):
                    byte_word_freqs[tuple(byte_word)] = freq
            except UnicodeEncodeError:
                continue

        next_id = 256
        while len(self.vocab) < vocab_size and next_id < 65536:  
            pair_freqs = defaultdict(int)
            for word, freq in byte_word_freqs.items():
                if len(word) < 2:
                    continue
                for i in range(len(word)-1):
                    pair = (word[i], word[i+1])
                    pair_freqs[pair] += freq
            
            if not pair_freqs:
                break
            valid_pairs = [(b1, b2) for b1, b2 in pair_freqs.keys() if 0 <= b1 <= 255 and 0 <= b2 <= 255]
            if not valid_pairs:
                break
                
            best_pair = max(valid_pairs, key=lambda x: pair_freqs[x])
  
            new_token = bytes([best_pair[0], best_pair[1]])

            if new_token not in self.reverse_vocab:
                self.vocab[next_id] = new_token
                self.reverse_vocab[new_token] = next_id
                self.merges.append((bytes([best_pair[0]]), bytes([best_pair[1]])))
                next_id += 1

            new_byte_word_freqs = {}
            for word, freq in byte_word_freqs.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word)-1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(next_id-1)  # 使用新token的ID
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_byte_word_freqs[tuple(new_word)] = freq
            
            byte_word_freqs = new_byte_word_freqs

        for token in self.special_tokens:
            try:
                token_bytes = token.encode('utf-8')
                if token_bytes not in self.reverse_vocab:
                    token_id = len(self.vocab)
                    self.vocab[token_id] = token_bytes
                    self.reverse_vocab[token_bytes] = token_id
            except UnicodeEncodeError:
                continue
        
        return self.vocab, self.merges

def _initialize_base_vocab(self):
        for i in range(256):
            token_bytes = bytes([i])
            self.vocab[i] = token_bytes
            self.reverse_vocab[token_bytes] = i

def _get_word_frequencies(self, text: str) -> Dict[Tuple[int, ...], int]:
        words = text.split()
        word_freqs = Counter()
        
        for word in tqdm.tqdm(words, desc="Processed word(s)"):
            byte_word = word.encode('utf-8')
            int_word = tuple(byte_word)
            word_freqs[int_word] += 1
            
        return dict(word_freqs)

def _get_pair_frequencies(self, word_freqs: Dict[Tuple[int, ...], int]) -> Dict[Tuple[int, int], int]:
        pair_freqs = defaultdict(int)
        
        for word_tuple, freq in word_freqs.items():
            if len(word_tuple) < 2:
                continue
                
            for i in range(len(word_tuple) - 1):
                pair = (word_tuple[i], word_tuple[i + 1])
                pair_freqs[pair] += freq
                
        return dict(pair_freqs)

def _merge_pair(self, pair: Tuple[int, int], word_freqs: Dict[Tuple[int, ...], int]):
        new_token = bytes(pair)

        if new_token in self.reverse_vocab:
            return
  
        new_token_id = len(self.vocab)
        self.vocab[new_token_id] = new_token
        self.reverse_vocab[new_token] = new_token_id
 
        updated_word_freqs = {}
        
        for word_tuple, freq in word_freqs.items():
            if len(word_tuple) < 2:
                updated_word_freqs[word_tuple] = freq
                continue
 
            i = 0
            new_word = []
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and (word_tuple[i], word_tuple[i + 1]) == pair:
                    new_word.append(new_token_id)
                    i += 2
                else:
                    new_word.append(word_tuple[i])
                    i += 1

            new_word_tuple = tuple(new_word)
            updated_word_freqs[new_word_tuple] = freq
            
        word_freqs.clear()
        word_freqs.update(updated_word_freqs)

def save(self, vocab_path: str, merges_path: str):

        with open(vocab_path, 'w', encoding='utf-8') as f:
            for token_id, token_bytes in self.vocab.items():
                try:
                    token_str = token_bytes.decode('utf-8', errors='replace')
                except:
                    token_str = str(token_bytes)
                f.write(f"{token_id}\t{token_str}\n")
        
        with open(merges_path, 'w', encoding='utf-8') as f:
            for idx, (b1, b2) in enumerate(self.merges):
                try:
                    b1_str = bytes([b1]).decode('utf-8', errors='replace')
                    b2_str = bytes([b2]).decode('utf-8', errors='replace')
                except:
                    b1_str = str(bytes([b1]))
                    b2_str = str(bytes([b2]))
                f.write(f"{idx}\t{b1_str}\t{b2_str}\n")
        
        print(f"Vocabulary saved to: {vocab_path}")
        print(f"Merge rules saved to: {merges_path}")

def run_train_bpe(input_path: str, vocab_size: int, special_tokens: List[str] = None):
    trainer = SafeBPETrainer(special_tokens=special_tokens)
    return trainer.train(input_path, vocab_size)

def Tokenizer__init__(self, vocab: Dict[int, bytes], merges: List[Tuple[bytes, bytes]], 
                 special_tokens: List[str] = None):
        self.vocab = vocab
        self.merges = merges
        self.special_tokens = special_tokens or []
        
        self.reverse_vocab = {token_bytes: token_id for token_id, token_bytes in vocab.items()}

        self.special_token_map = {}
        for token in special_tokens:
            token_bytes = token.encode('utf-8')
            if token_bytes in self.reverse_vocab:
                self.special_token_map[token] = self.reverse_vocab[token_bytes]
    
def from_files(cls, vocab_filepath: str, merges_filepath: str, 
                   special_tokens: List[str] = None):

        with open(vocab_filepath, 'r', encoding='utf-8') as f:
            vocab_json = json.load(f)
        
        vocab = {}
        for token_id_str, token_repr in vocab_json.items():
            token_id = int(token_id_str)
            if isinstance(token_repr, str):
                token_bytes = token_repr.encode('utf-8')
            elif isinstance(token_repr, list):
                token_bytes = bytes(token_repr)
            else:
                continue
            vocab[token_id] = token_bytes
        
        with open(merges_filepath, 'r', encoding='utf-8') as f:
            merges_json = json.load(f)
        
        merges = []
        for merge_pair in merges_json:
            if len(merge_pair) >= 2:
                b1_repr, b2_repr = merge_pair[0], merge_pair[1]
                
                if isinstance(b1_repr, str):
                    b1 = b1_repr.encode('utf-8')
                elif isinstance(b1_repr, list):
                    b1 = bytes(b1_repr)
                else:
                    continue
                
                if isinstance(b2_repr, str):
                    b2 = b2_repr.encode('utf-8')
                elif isinstance(b2_repr, list):
                    b2 = bytes(b2_repr)
                else:
                    continue
                
                merges.append((b1, b2))
        
        return cls(vocab, merges, special_tokens)
    
def encode(self, text: str) -> List[int]:
        if self.special_token_map:
            for token, token_id in self.special_token_map.items():
                if token in text:
                    parts = text.split(token)
                    result = []
                    for i, part in enumerate(parts):
                        if part:
                            result.extend(self._encode_text(part))
                        if i < len(parts) - 1:
                            result.append(token_id)
                    return result
        
        return self._encode_text(text)
    
def _encode_text(self, text: str) -> List[int]:

        byte_text = text.encode('utf-8')
        tokens = [bytes([b]) for b in byte_text]
  
        for b1, b2 in self.merges:
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == b1 and tokens[i + 1] == b2:
                    new_tokens.append(b1 + b2)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        token_ids = []
        for token in tokens:
            if token in self.reverse_vocab:
                token_ids.append(self.reverse_vocab[token])
            else:
                for b in token:
                    byte_token = bytes([b])
                    token_ids.append(self.reverse_vocab[byte_token])
        
        return token_ids
    
def decode(self, ids: List[int]) -> str:
        byte_sequence = b''
        for token_id in ids:
            if token_id in self.vocab:
                byte_sequence += self.vocab[token_id]
            else:
                continue
        
        try:
            return byte_sequence.decode('utf-8')
        except:
            return byte_sequence.decode('utf-8', errors='replace')
    
def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:

        for text in iterable:
            for token_id in self.encode(text):
                yield token_id

def run_experiment():
    print("\n" + "="*60)
    print("Tokenizer testing")
    print("="*60)
    
    print("\n1. Sampling 10 documents from the dataset...")
    
    documents = []
    with open(data_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
  
    lines = [line.strip() for line in lines if line.strip()]
    
    if len(lines) >= 10:
        import random
        sampled_indices = random.sample(range(len(lines)), 10)
        documents = [lines[i] for i in sampled_indices]
    else:
        documents = lines[:10]
    
    print(f"   Sampled {len(documents)} document(s)")

    print("\n2. Encoding documents and calculating compression ratio:")
    print("-"*80)
    print(f"{'documents':<5} {'Raw bytes':<10} {'Tokens':<10} {'byte/Token':<12} {'Compression ratio':<10}")
    print("-"*80)
    
    total_bytes = 0
    total_tokens = 0
    
    for i, doc in enumerate(documents):
        if not doc:
            continue
            
        encoded = tokenizer.encode(doc)
        
        if not encoded:
            continue
            
        original_bytes = len(doc.encode('utf-8'))
        token_count = len(encoded)
        bytes_per_token = original_bytes / token_count
        compression_ratio = bytes_per_token
        
        total_bytes += original_bytes
        total_tokens += token_count
        
        print(f"{i+1:<5} {original_bytes:<10} {token_count:<10} {bytes_per_token:<12.2f} {compression_ratio:<10.2f}")
        
        if i < 3:
            print(f"    sample: {doc[:60]}...")
            print(f"    Tokens: {encoded[:15]}..." if len(encoded) > 15 else f"    Tokens: {encoded}")
    print("-"*80)
    if total_tokens > 0:
        avg_bytes_per_token = total_bytes / total_tokens
        print(f"{'total':<5} {total_bytes:<10} {total_tokens:<10} {avg_bytes_per_token:<12.2f} {avg_bytes_per_token:<10.2f}")
        
        print(f"\nTesting conclusion:")
        print(f"  Average compression ratio: {avg_bytes_per_token:.2f} bytes/token")
        print(f"  Average characters per token {avg_bytes_per_token:.2f} raw byte(s)")
        print(f"  Tokenization reduced the sequence length by {((1 - 1/avg_bytes_per_token)*100):.1f}%" if avg_bytes_per_token > 1 else "  压缩率小于1，可能需要调整词汇表大小")
    
    return documents

experiment_docs = run_experiment()


Overwriting submission.py


In [3]:
import sys
import os
import pytest

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
os.chdir(project_root)

# Please CHANGE submission_dir to the directory of your answer.ipynb file, below is an example
submission_dir = os.path.join(project_root, "answers/yuanhao")
# submission_dir = os.path.join(project_root, "answers/xxxx")
if submission_dir not in sys.path:
    sys.path.insert(0, submission_dir)

# RUN TEST
# Args description:
# "-v": Verbose output
# "tests": Points to the test directory
# "-k sink": Run only tests related to "sink attention" (optional, for speed)
args = [
    "-v",
    "tests", 
    "-k", "test_attention_with_sink"
]

print(f"Current Working Directory: {os.getcwd()}")
pytest.main(args)

Current Working Directory: d:\
============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-9.0.2, pluggy-1.6.0 -- d:\juliannnnnn_project\venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: d:\
plugins: jaxtyping-0.3.6
collecting ... collected 0 items

============================ no tests ran in 0.00s ============================


ERROR: file or directory not found: tests



<ExitCode.USAGE_ERROR: 4>